# Stage 9 — Conclusions

Synthesize findings across all stages. Answer research questions.

**Regenerate report:** `python scripts/generate_report.py` → `outputs/final_report.md`

In [9]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
from config import OUTPUTS_DIR

## 1. Did feature fusion add value?

In [10]:
ab_path = OUTPUTS_DIR / "fusion_ab_results.json"
if ab_path.exists():
    with open(ab_path) as f:
        ab = json.load(f)
    a = ab["analysis"]
    print(f"HF Test AUC:      {a['test_auc_hf']:.4f}")
    print(f"Acoustic Test AUC: {a['test_auc_acoustic']:.4f}")
    print(f"Fusion Test AUC:  {a['test_auc_fusion']:.4f}")
    print(f"Fusion − HF:      {a['fusion_minus_hf']:+.4f}")
    print(f"\n{a['conclusion']}")
else:
    print("Run: python scripts/experiment_fusion_ab.py")

HF Test AUC:      0.9039
Acoustic Test AUC: 0.6873
Fusion Test AUC:  0.9052
Fusion − HF:      +0.0013

Fusion ≈ HuBERT. Transformer likely encodes most acoustic info. FUSION MARGINAL.


## 2. Model comparison (clean)

In [11]:
sources = [
    ("hubert_base_ls960", "transformer_results.json", lambda d: d.get("hubert_base_ls960", {}).get("auc_roc", 0)),
    ("Wav2Vec2", "transformer_results.json", lambda d: d.get("wav2vec2", {}).get("auc_roc", 0)),
    ("WavLM", "transformer_results.json", lambda d: d.get("wavlm", {}).get("auc_roc", 0)),
    ("Whisper", "transformer_results.json", lambda d: d.get("whisper", {}).get("auc_roc", 0)),
    ("Fusion (A/B)", "fusion_ab_results.json", lambda d: d["analysis"]["test_auc_fusion"]),
    ("CNN", "cnn_results.json", lambda d: d.get("auc_roc", 0)),
    ("RF", "baseline_results.json", lambda d: d.get("random_forest", {}).get("auc_roc", 0)),
]

clean_aucs = []
for name, fname, getter in sources:
    p = OUTPUTS_DIR / fname
    if p.exists():
        with open(p) as f:
            d = json.load(f)
        try:
            auc = getter(d)
            clean_aucs.append((name, auc))
        except (KeyError, TypeError):
            pass

for name, auc in sorted(clean_aucs, key=lambda x: -x[1]):
    print(f"{name}: {auc:.4f}")

hubert_base_ls960: 0.9527
Wav2Vec2: 0.9463
WavLM: 0.9099
Fusion (A/B): 0.9052
Whisper: 0.7951
CNN: 0.6970
RF: 0.6277


## 3. Noise robustness

In [12]:
nr_path = OUTPUTS_DIR / "noise_robustness_results.json"
if nr_path.exists():
    with open(nr_path) as f:
        nr = json.load(f)
    print("Model degradation at 0 dB SNR:")
    for model_name, model_data in nr.items():
        for noise_type in ["white", "pink", "compression"]:
            if noise_type in model_data and "0dB" in model_data[noise_type]:
                auc = model_data[noise_type]["0dB"]["auc"]
                print(f"  {model_name} ({noise_type}): AUC={auc:.3f}")
else:
    print("Run: python scripts/run_noise_robustness.py")

Model degradation at 0 dB SNR:
  hubert_base_ls960 (white): AUC=0.808
  hubert_base_ls960 (pink): AUC=0.731
  hubert_base_ls960 (compression): AUC=0.935
  wav2vec2 (white): AUC=0.305
  wav2vec2 (pink): AUC=0.346
  wav2vec2 (compression): AUC=0.924
  wavlm (white): AUC=0.812
  wavlm (pink): AUC=0.669
  wavlm (compression): AUC=0.900
  whisper (white): AUC=0.547
  whisper (pink): AUC=0.514
  whisper (compression): AUC=0.810
  cnn (white): AUC=0.403
  cnn (pink): AUC=0.408
  cnn (compression): AUC=0.707


## 4. Research questions answered

- **Q1:** Which acoustic cues best discriminate real vs synthetic? *(See Stage 3 EDA)*
- **Q2:** Do transformers outperform classical/CNN baselines? *(See comparison above)*
- **Q3:** Does feature fusion add value beyond transformers? *(See fusion A/B)*
- **Q4:** How does performance degrade under noise? *(See Stage 8)*

## 5. Limitations & future work

- Dataset: single language, limited TTS coverage
- Speaker-disjoint split: test speakers unseen
- Future: multilingual, adversarial robustness, online detection